[Back to NLP guideline](Natural-Language-Processing.html)


## **Modeling in NLP**
NLP modeling is the process of designing a function that maps language input to useful output. The input may be a sentence, a document, a pair of sentences, or a prompt. The output may be a label, a label sequence, a generated sequence, a score, or a probability distribution over the next token.

The key idea is that a model is not just a black box placed after preprocessing. It defines what kind of linguistic evidence can be used. A linear model mostly sees weighted features. An RNN sees a sequence through a recurrent memory. A Transformer lets tokens directly compare with other tokens. So when we choose a model, we are also choosing a way for the system to represent context.

```text
Raw Text -> Tokenisation -> Representation -> Model -> Prediction / Generation
```

A useful way to study NLP models is not to memorize model names first, but to ask:

- What is the input?
- What is the output?
- What probability is the model trying to estimate?
- How much context can the architecture use?
- How is the final output decoded?

### **What Does Modeling Mean in NLP?**
In NLP, a model is a parameterized function that scores or predicts linguistic structures. The word *parameterized* means the model contains learnable values, such as weights in a linear classifier or matrices inside a Transformer. Training changes these parameters so that correct outputs receive higher scores than incorrect outputs.

A useful mental model is: the representation gives the model raw materials, and the model decides how to combine them. If the representation contains word counts, the model learns which words matter. If the representation contains token embeddings, the model learns how these token meanings interact.

For a classification task, the model may estimate:

$$
P(y \mid x)
$$

where $x$ is the input text and $y$ is the output label.

For a generation task, the model may estimate:

$$
P(y_1, y_2, ..., y_T \mid x)
$$

where the output is a sequence of tokens.

A model usually contains three conceptual parts:

| Part | Role | Example |
|---|---|---|
| Input representation | Turns text into vectors | BoW, TF-IDF, embeddings, contextual embeddings |
| Context modeling architecture | Combines information across tokens | Linear model, CNN, RNN, Transformer |
| Prediction head | Converts hidden representation into output | Classification head, token classifier, LM head |

The same architecture can be used for different tasks by changing the prediction head. For example, a Transformer encoder can be used for sentiment classification, token classification, retrieval, or span extraction.

```text
Representation answers: what does each token/document look like numerically?
Architecture answers: how does information move between tokens?
Prediction head answers: what kind of output do we want?
```

### **Input-Output Formulations in NLP**
Before choosing a model architecture, it is important to define the input-output structure of the task. Many NLP tasks can be grouped by the shape of their input and output. This step is simple but important: if the output is one label, a classifier is enough; if the output is one label per token, the model must keep token-level representations; if the output is a new sentence, the model also needs a decoding process.

| Formulation | Input | Output | Example Task |
|---|---|---|---|
| Sequence-to-Label | one sequence | one label | sentiment classification |
| Sequence-to-Label-Sequence | one sequence | one label per token | POS tagging, NER |
| Sequence-to-Sequence | one sequence | another sequence | translation, summarisation |
| Language Modeling | previous tokens | next token / continuation | GPT-style generation |
| Pairwise / Matching | two sequences | score or label | NLI, semantic similarity, retrieval |

#### **Sequence-to-Label Modeling**
Sequence-to-label modeling maps an entire input sequence to a single output label. The central challenge is compression: the model must turn a variable-length text into one fixed-size decision signal. A short review and a long review both need to become one final prediction.

```text
Input:  "This movie was surprisingly good"
Output: Positive
```

The model needs to compress the whole sequence into one representation. This can be done by:

- averaging word embeddings
- using the final hidden state of an RNN
- using a `[CLS]` representation in BERT-style models
- pooling token representations from a Transformer

Common tasks:

| Task | Input | Label |
|---|---|---|
| Sentiment classification | review text | positive / negative |
| Spam detection | email text | spam / not spam |
| Topic classification | news article | politics / sport / finance |
| Intent detection | user utterance | book_flight / cancel_order |

A typical probability formulation is:

$$
\hat{y} = \arg\max_y P(y \mid x)
$$

<details>
<summary>Python Sequence-to-Label with TF-IDF + Logistic Regression</summary>

```python
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

texts = [
    "the movie was excellent",
    "great story and acting",
    "the film was boring",
    "terrible pacing and weak plot"
]
labels = [1, 1, 0, 0]

model = Pipeline([
    ("tfidf", TfidfVectorizer()),
    ("clf", LogisticRegression())
])

model.fit(texts, labels)
print(model.predict(["excellent acting"]))
```
</details>

#### **Sequence-to-Label-Sequence Modeling**
Sequence-to-label-sequence modeling predicts one label for each input token. Unlike document classification, the model cannot only build one sentence-level vector. It must preserve information at each token position because every token needs its own prediction.

```text
Input:  John   works  at  Google
Output: B-PER  O      O   B-ORG
```

This is also called **sequence labeling** or **token classification**.

Common tasks:

| Task | Output Label Meaning |
|---|---|
| POS tagging | grammatical category for each token |
| Named Entity Recognition | entity span and type |
| Chunking | phrase boundary |
| Slot filling | semantic slot in a user utterance |

For an input sequence $x_1, ..., x_n$, the model predicts:

$$
y_1, ..., y_n
$$

Usually the input length and output length are the same.

> Image for sequence labeling:
>
> ![Sequence labeling example](assets/sequence-labeling-example.svg)

<details>
<summary>Python Token Classification Output Shape</summary>

```python
import torch
import torch.nn as nn

batch_size = 2
seq_len = 5
hidden_dim = 16
num_tags = 4

hidden_states = torch.randn(batch_size, seq_len, hidden_dim)
classifier = nn.Linear(hidden_dim, num_tags)

logits = classifier(hidden_states)
print(logits.shape)  # [batch_size, seq_len, num_tags]
```
</details>

#### **Sequence-to-Sequence Modeling**
Sequence-to-sequence modeling maps one input sequence to another output sequence. The output length may be different from the input length. This formulation is more difficult than classification because the model must decide both content and order. A translation system, for example, must understand the source sentence and then produce a fluent target sentence with a different word order.

```text
Input:  I love natural language processing
Output: J'aime le traitement du langage naturel
```

Typical tasks:

| Task | Input | Output |
|---|---|---|
| Machine Translation | sentence in source language | sentence in target language |
| Summarisation | long document | short summary |
| Question Answering | question + passage | answer text |
| Text-to-SQL | user question | SQL query |

The model estimates:

$$
P(y_1, ..., y_m \mid x_1, ..., x_n)
$$

This is usually factorized autoregressively:

$$
P(y \mid x) = \prod_{t=1}^{m} P(y_t \mid y_{<t}, x)
$$

> Image for seq2seq with attention:
>
> ![Seq2Seq RNN with attention](https://upload.wikimedia.org/wikipedia/commons/c/c7/Seq2seq_RNN_encoder-decoder_with_attention_mechanism%2C_training.png)
>
> Source: [Wikipedia - Recurrent neural network](https://en.wikipedia.org/wiki/Recurrent_neural_network)

#### **Language Modeling**
Language modeling predicts the next token given previous tokens. This looks simple, but it creates a very rich learning signal: every position in a text can become a training example. By learning to predict continuations, the model also learns grammar, common facts, discourse patterns, and many surface regularities of language.

```text
Input:  Natural language processing is
Output: [fun, difficult, useful, ...]
```

A language model estimates the probability of a token sequence:

$$
P(w_1, w_2, ..., w_n)
$$

Using the chain rule:

$$
P(w_1, ..., w_n) = \prod_{i=1}^{n} P(w_i \mid w_1, ..., w_{i-1})
$$

In modern decoder-only Transformers, this becomes **next-token prediction**:

$$
P(w_i \mid w_{<i})
$$

Language models are used for:

- text generation
- autocomplete
- chatbots
- code generation
- scoring sentence likelihood
- pretraining large neural models

<details>
<summary>Python Next-Token Prediction with transformers</summary>

```python
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

model_name = "gpt2"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

prompt = "Natural language processing is"
inputs = tokenizer(prompt, return_tensors="pt")

with torch.no_grad():
    outputs = model(**inputs)

next_token_logits = outputs.logits[0, -1]
next_token_id = next_token_logits.argmax().item()

print(tokenizer.decode([next_token_id]))
```
</details>

#### **Pairwise and Matching-Based Modeling**
Pairwise modeling takes two pieces of text and predicts a relation or similarity score. This is common in search and reasoning tasks, where meaning depends on the relationship between two texts rather than one text alone.

```text
Text A: A dog is running in the park.
Text B: An animal is moving outdoors.
Output: entailment / high similarity
```

Common tasks:

| Task | Input Pair | Output |
|---|---|---|
| Natural Language Inference | premise + hypothesis | entailment / contradiction / neutral |
| Semantic Textual Similarity | sentence A + sentence B | similarity score |
| Question Answering Retrieval | query + passage | relevance score |
| Duplicate Detection | question A + question B | duplicate / not duplicate |

There are two common modeling styles:

| Style | Method | Strength |
|---|---|---|
| Cross-encoder | encode both texts together | high accuracy, full interaction |
| Bi-encoder | encode separately, compare vectors | efficient retrieval |

### **Probabilistic View of NLP Modeling**
Many NLP models can be understood through probability. The model either predicts labels given text, generates text, or generates output conditioned on input. This probabilistic view is helpful because it connects many different tasks under one language: the model assigns high probability to desired outputs and low probability to undesired outputs.

#### **Discriminative Modeling**
Discriminative models directly model the decision boundary between possible outputs. Instead of explaining how the input text was generated, they focus on choosing the correct label for the given input. In practice, many supervised NLP systems are discriminative because the final goal is accurate prediction.

Discriminative models directly model:

$$
P(y \mid x)
$$

They learn a boundary between output classes.

Examples:

| Model | Task |
|---|---|
| Logistic Regression | sentiment classification |
| Linear SVM | document classification |
| BERT classifier | intent detection |
| CRF tagger | sequence labeling |

Discriminative models are usually preferred when the goal is prediction and labelled data is available.

#### **Generative Modeling**
Generative models model the data generation process. They try to describe how text and labels could have been produced. This can be useful when we want to generate new text, estimate likelihood, or use assumptions about how words appear under different classes.

In classification, a generative model may estimate:

$$
P(x, y) = P(y)P(x \mid y)
$$

Then prediction uses Bayes' rule:

$$
P(y \mid x) = \frac{P(x \mid y)P(y)}{P(x)}
$$

Naive Bayes is a classic example. For a document $d$ and class $c$:

$$
P(c \mid d) \propto P(c) \prod_i P(w_i \mid c)
$$

Generative language models also estimate how likely text is:

$$
P(w_1, ..., w_n)
$$

#### **Conditional Generation**
Conditional generation models generate output text given input text. It sits between classification and pure language modeling: the model is generative because it produces a sequence, but conditional because the generated sequence should depend on a given input.

Conditional generation models generate output text given input text:

$$
P(y \mid x)
$$

But unlike simple classification, $y$ is a sequence.

Examples:

| Task | Conditional Probability |
|---|---|
| Translation | $P(\text{target sentence} \mid \text{source sentence})$ |
| Summarisation | $P(\text{summary} \mid \text{document})$ |
| Dialogue | $P(\text{response} \mid \text{conversation})$ |

Conditional generation usually requires decoding because the model must choose a sequence of output tokens.

### **Architecture as Context Modeling**
An NLP architecture defines how the model moves information across tokens. The main difference between model families is their **context modeling strategy**. This is the heart of modeling: language meaning often depends on context, so architectures differ mainly in how they let one token influence another.

| Architecture | Context Strategy | Good At | Weakness |
|---|---|---|---|
| Linear Models | no real token-token context unless features encode it | fast baselines | weak word order modeling |
| CNN-Based Models | local n-gram windows | local phrase patterns | limited long-range context |
| RNN / LSTM / GRU | recurrent hidden state | sequential context | hard to parallelise |
| Transformer-Based Models | self-attention between tokens | global context | quadratic attention cost |

#### **Linear Models**
Linear models score text using weighted features. They are simple but still important because they establish a strong baseline. If a linear model performs well, the task may be mostly solvable from surface lexical cues. If it fails, the task may require word order, syntax, or deeper context modeling.

Linear models score text using weighted features:

$$
f(x)=w^Tx+b
$$

In traditional NLP, $x$ is often BoW, TF-IDF, or n-gram features.

```text
Input sentence: "not very good"
Features: not=1, very=1, good=1, not_good=1
Model score: sum(feature value * feature weight)
```

Linear models are not context-aware by themselves. Context must be manually added through features such as:

- bigrams
- trigrams
- negation features
- lexicon features
- syntactic features

<details>
<summary>Python Linear Model with TF-IDF</summary>

```python
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

model = Pipeline([
    ("tfidf", TfidfVectorizer(ngram_range=(1, 2))),
    ("clf", LogisticRegression())
])
```
</details>

#### **CNN-Based Models**
CNNs are often associated with images, but they can also be used for text. In text CNNs, convolution filters slide over word embeddings and detect local n-gram-like patterns. The model is not reading the full sentence at once; it is scanning for useful local patterns that strongly indicate the class.

```text
Embeddings -> Convolution Filters -> Max Pooling -> Classifier
```

For example, a filter with width 3 may detect useful phrase patterns such as:

```text
not very good
really loved this
waste of time
```

> Image for CNN-based text modeling:
>
> ![CNN convolutional layers](https://upload.wikimedia.org/wikipedia/commons/5/51/CNN_Convolutional_Layers.svg)
>
> Source: [Wikimedia Commons - CNN Convolutional Layers](https://commons.wikimedia.org/wiki/File:CNN_Convolutional_Layers.svg)

| Component | NLP Interpretation |
|---|---|
| Embedding matrix | sentence represented as token vectors |
| Convolution filter | detects local phrase pattern |
| Feature map | where the pattern appears |
| Max pooling | keeps the strongest signal |
| Classifier | predicts the final label |

CNNs are useful when local phrases are strong indicators, but they are less natural for long-range dependencies.

<details>
<summary>Python Text CNN Sketch</summary>

```python
import torch
import torch.nn as nn
import torch.nn.functional as F

class TextCNN(nn.Module):
    def __init__(self, vocab_size, embed_dim, num_classes):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.conv = nn.Conv1d(
            in_channels=embed_dim,
            out_channels=100,
            kernel_size=3
        )
        self.fc = nn.Linear(100, num_classes)

    def forward(self, token_ids):
        x = self.embedding(token_ids)      # [batch, seq_len, embed_dim]
        x = x.transpose(1, 2)              # [batch, embed_dim, seq_len]
        x = F.relu(self.conv(x))           # [batch, channels, new_seq_len]
        x = F.max_pool1d(x, kernel_size=x.size(2)).squeeze(2)
        return self.fc(x)
```
</details>

#### **RNN / LSTM / GRU**
RNNs process text sequentially. At each time step, the model updates a hidden state using the current token and the previous hidden state. This hidden state is the model's running summary of what it has read so far, so RNNs naturally match the left-to-right structure of language.

$$
h_t = \tanh(Wx_t + Uh_{t-1} + b)
$$

The hidden state acts as a memory of previous context.

> Image for basic RNN:
>
> ![RNN unfolded](https://upload.wikimedia.org/wikipedia/commons/b/b5/Recurrent_neural_network_unfold.svg)
>
> Source: [Wikipedia - Recurrent Neural Network](https://en.wikipedia.org/wiki/Recurrent_neural_network)

RNNs can be used for:

| Pattern | Input | Output | Example |
|---|---|---|---|
| many-to-one | token sequence | one label | sentiment classification |
| many-to-many | token sequence | token label sequence | NER, POS tagging |
| encoder-decoder | source sequence | generated target sequence | translation |

Basic RNNs have difficulty with long-range dependencies because gradients must travel through many time steps.

LSTM and GRU are gated variants that reduce this problem.

> Image for LSTM:
>
> ![LSTM unit](https://upload.wikimedia.org/wikipedia/commons/6/63/Long_Short-Term_Memory.svg)
>
> Source: [Wikipedia - Recurrent Neural Network](https://en.wikipedia.org/wiki/Recurrent_neural_network)

| Model | Main Mechanism | Intuition |
|---|---|---|
| RNN | one hidden state | simple sequential memory |
| LSTM | cell state + input/forget/output gates | stronger long-term memory |
| GRU | update/reset gates | simpler gated memory |

<details>
<summary>Python BiLSTM Token Classifier</summary>

```python
import torch
import torch.nn as nn

class BiLSTMTagger(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, num_tags):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.lstm = nn.LSTM(
            embed_dim,
            hidden_dim,
            batch_first=True,
            bidirectional=True
        )
        self.classifier = nn.Linear(hidden_dim * 2, num_tags)

    def forward(self, token_ids):
        embeddings = self.embedding(token_ids)
        outputs, _ = self.lstm(embeddings)
        return self.classifier(outputs)

model = BiLSTMTagger(10000, 128, 64, 5)
batch = torch.tensor([[11, 22, 33, 44]])
print(model(batch).shape)  # [batch, seq_len, num_tags]
```
</details>

#### **Transformer-Based Models**
Transformers model context using self-attention. Instead of processing tokens one by one, each token can directly attend to other tokens in the sequence. This gives the model a shorter path for long-range dependencies: two distant words can interact in one attention layer instead of passing information through many recurrent steps.

For each token representation $x_i$, the model creates:

$$
q_i = W_Qx_i, \quad k_i = W_Kx_i, \quad v_i = W_Vx_i
$$

Scaled dot-product attention is:

$$
\text{Attention}(Q,K,V)=\text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V
$$

> Image for Transformer architecture:
>
> ![Transformer full architecture](https://upload.wikimedia.org/wikipedia/commons/3/34/Transformer%2C_full_architecture.png)
>
> Source: [Wikipedia - Transformer](https://en.wikipedia.org/wiki/Transformer_(deep_learning))

A Transformer block usually contains:

| Component | Role |
|---|---|
| Multi-head self-attention | lets tokens exchange contextual information |
| Feed-forward network | applies non-linear transformation at each position |
| Residual connection | stabilizes deep networks |
| Layer normalization | normalizes hidden activations |
| Positional encoding | injects word order information |

Transformer model families:

| Family | Attention Pattern | Suitable Tasks |
|---|---|---|
| Encoder-only | bidirectional attention | classification, NER, embeddings |
| Decoder-only | causal attention | text generation, chat, completion |
| Encoder-decoder | encoder attention + decoder cross-attention | translation, summarisation |

<details>
<summary>Python Multi-Head Attention</summary>

```python
import torch
import torch.nn as nn

attention = nn.MultiheadAttention(
    embed_dim=128,
    num_heads=8,
    batch_first=True
)

x = torch.randn(2, 10, 128)
output, weights = attention(x, x, x)

print(output.shape)   # [2, 10, 128]
print(weights.shape)  # [2, 10, 10]
```
</details>

### **Prediction Heads and Output Structures**
A prediction head converts hidden representations into the required output structure. The same backbone can support multiple heads. This separation explains why pretrained models are flexible: the backbone learns general language representations, while the head adapts those representations to a specific task.

```text
Backbone hidden states -> Prediction Head -> Task-specific Output
```

#### **Classification Head**
A classification head maps one sequence representation to class logits.

For an encoder model, this often uses the `[CLS]` vector:

$$
z = W h_{CLS} + b
$$

$$
P(y \mid x)=\text{softmax}(z)
$$

Used for:

- sentiment classification
- topic classification
- intent detection
- natural language inference

<details>
<summary>Python Classification Head</summary>

```python
import torch
import torch.nn as nn

hidden_size = 768
num_classes = 3

cls_vector = torch.randn(4, hidden_size)  # [batch, hidden]
head = nn.Linear(hidden_size, num_classes)

logits = head(cls_vector)
print(logits.shape)  # [4, 3]
```
</details>

#### **Token Classification Head**
A token classification head maps each token representation to a label distribution.

$$
z_t = W h_t + b
$$

Used for:

- POS tagging
- NER
- chunking
- slot filling

```text
Input hidden states: [h_1, h_2, ..., h_n]
Output logits:       [z_1, z_2, ..., z_n]
```

#### **Language Modeling Head**
A language modeling head maps a hidden state to vocabulary logits.

$$
z_t = W h_t + b
$$

$$
P(w_{t+1} \mid w_{\le t})=\text{softmax}(z_t)
$$

The output dimension is the vocabulary size.

```text
hidden state -> vocabulary logits -> probability over all tokens
```

#### **Encoder-Decoder Output Layer**
In encoder-decoder models, the decoder produces one hidden state per generated position. Each decoder hidden state is mapped to vocabulary logits.

```text
encoder input -> encoder states
previous output tokens -> decoder states -> vocabulary distribution
```

The decoder may also attend to encoder states through cross-attention.

### **Decoding and Inference**
For classification, inference is often simple: choose the label with the highest score.

For generation, inference is more complex because the model must choose an entire output sequence. A generation model does not directly output one finished answer; it repeatedly outputs a distribution over the next token. Decoding is the procedure that turns those step-by-step distributions into final text.

At each decoding step, the model produces a probability distribution over the vocabulary:

$$
P(y_t \mid y_{<t}, x)
$$

The decoding algorithm decides which token to choose next.

#### **Greedy Decoding**
Greedy decoding chooses the highest-probability token at each step.

$$
y_t = \arg\max_w P(w \mid y_{<t}, x)
$$

Example:

```text
Step 1: choose "I"
Step 2: choose "love"
Step 3: choose "NLP"
```

Pros:

- simple
- fast
- deterministic

Cons:

- local choices may be globally bad
- often produces generic output
- cannot recover from early mistakes

<details>
<summary>Python Greedy Decoding Sketch</summary>

```python
def greedy_decode(next_token_fn, start_token, end_token, max_len=20):
    output = [start_token]

    for _ in range(max_len):
        probs = next_token_fn(output)
        next_token = max(probs, key=probs.get)
        output.append(next_token)

        if next_token == end_token:
            break

    return output
```
</details>

#### **Beam Search**
Beam search keeps the top $k$ partial sequences at each decoding step. The value $k$ is called the beam width.

> Image for beam search:
>
> ![Beam search](https://upload.wikimedia.org/wikipedia/commons/2/23/Beam_search.gif)
>
> Source: [Wikipedia - Beam Search](https://en.wikipedia.org/wiki/Beam_search)

```text
beam width = 3
keep only the best 3 partial sequences after each step
```

Pros:

- explores more options than greedy decoding
- often improves translation and summarisation quality
- deterministic if no sampling is used

Cons:

- slower than greedy decoding
- large beam width may produce bland output
- still not guaranteed to find the true global optimum

A common score is the sum of log probabilities:

$$
score(y)=\sum_{t=1}^{T}\log P(y_t \mid y_{<t},x)
$$

Length normalization is often added because raw log probability tends to prefer short sequences.

#### **Sampling-Based Decoding**
Sampling-based decoding randomly samples from the model's probability distribution.

Instead of always taking the maximum probability token, it samples:

$$
y_t \sim P(\cdot \mid y_{<t}, x)
$$

Common strategies:

| Method | Idea | Effect |
|---|---|---|
| Temperature | divide logits by temperature | controls randomness |
| Top-k Sampling | sample only from top k tokens | removes very unlikely tokens |
| Top-p Sampling | sample from smallest set with cumulative probability p | adapts candidate set size |

Temperature modifies logits before softmax:

$$
P(w)=\text{softmax}(z_w / T)
$$

| Temperature | Behavior |
|---|---|
| Low temperature | safer, more deterministic |
| High temperature | more diverse, more risky |

Sampling is widely used in open-ended generation because there may be many valid continuations.

### **Choosing a Modeling Strategy**
Choosing an NLP model depends on the task formulation, data size, compute budget, latency requirement, and interpretability requirement. There is no single best model for every NLP task. A small business classifier may benefit more from a transparent TF-IDF baseline than from a large neural model, while open-ended generation requires an autoregressive language model.

| Situation | Recommended Starting Point | Reason |
|---|---|---|
| Small labelled classification dataset | TF-IDF + Logistic Regression | strong and interpretable baseline |
| Local phrase patterns matter | Text CNN | captures n-gram-like features |
| Token-level sequence labels | BiLSTM or encoder Transformer | uses context around each token |
| Translation / summarisation | Encoder-decoder Transformer | input and output are both sequences |
| Open-ended generation | Decoder-only Transformer | natural next-token generation |
| Retrieval at scale | Bi-encoder | efficient vector search |
| Highest accuracy on pair classification | Cross-encoder | full interaction between two texts |

A practical modeling workflow:

```text
1. Define input-output formulation
2. Build a simple baseline
3. Check failure cases
4. Choose architecture based on context requirement
5. Choose prediction head
6. Choose decoding strategy if generation is needed
7. Evaluate with task-appropriate metrics
```

Modeling in NLP is therefore not just about choosing a famous architecture. It is about matching the task structure to the right probability view, context mechanism, output head, and inference method.
